# Foundational Models

Learn how to initialise chat models and agents in LangChain, customise generation settings, switch providers, and stream responses token-by-token.

**Key learnings**
- `init_chat_model` gives a provider-agnostic way to create chat models
- Agents wrap a model so you can invoke them with message lists
- Streaming (`stream_mode="messages"`) yields tokens as they are generated


In [ ]:
from dotenv import load_dotenv

# Load API keys from the project .env file into the process environment
load_dotenv()


## Initialising and invoking a model

`init_chat_model` creates a chat model from a model id string. `invoke` sends one prompt and returns an AI message object with `.content` and `.response_metadata`.


In [ ]:
from langchain.chat_models import init_chat_model

# Provider-agnostic constructor — model id selects the backend
model = init_chat_model(model="gpt-5-nano")


In [ ]:
response = model.invoke("What's the capital of the Moon?")

response

In [ ]:
print(response.content)

In [ ]:
from pprint import pprint

pprint(response.response_metadata)

## Customising your Model

Provider-specific kwargs (like `temperature`) are passed through to the underlying model. Higher temperature usually means more varied, less deterministic answers.


In [ ]:
model = init_chat_model(
    model="gpt-5-nano",
    # Kwargs passed to the model:
    temperature=1.0
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

## Model Providers

LangChain supports many chat providers through the same interface. See the [chat model integrations docs](https://docs.langchain.com/oss/python/integrations/chat).

Below: Anthropic via `init_chat_model`, and Google Gemini via the dedicated `ChatGoogleGenerativeAI` class (requires the matching API keys in `.env`).


In [ ]:
model = init_chat_model(model="claude-sonnet-4-6")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

## Initialising and invoking an agent

`create_agent` builds a tool-calling agent around a model. You can pass a model object, a model id string, or a shorthand string. Invoke with a `messages` list of LangChain message objects.


In [ ]:
from langchain.agents import create_agent

# Agent wraps the model; tools/system prompt can be added later
agent = create_agent(model=model)


In [ ]:
agent = create_agent(model="claude-sonnet-4-6")

In [ ]:
agent = create_agent("gpt-5-nano")

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response['messages'][-1].content)

### Multi-turn message history

Pass prior `HumanMessage` / `AIMessage` pairs so the agent continues the conversation with full context.


In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)

## Streaming Output

`agent.stream(..., stream_mode="messages")` yields `(token, metadata)` pairs. Print `token.content` as it arrives for a live typing effect.


In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token